# 00 · What can skeleton history add?


We want to know whether an explicit skeleton helps a small predictor after it
has already seen the recent video. A skeleton is derived from the video, so
the claim concerns useful representation under these predictors. It does not
claim that pose reveals information absent from RGB.

Follow one walking clip. The predictor sees frames 0–31. A frozen video teacher
supplies a target at frames 38–39. This target is a vector of numbers describing
the person region. We predict those numbers, not a diagnosis or future pixels.

**Run this notebook independently in a fresh kernel.** The default
`teach` mode uses small generated examples. Set `FI_TUTORIAL_MODE=inspect`
and `FI_RUN_ROOT` before starting the kernel to read saved artifacts.
Set `FI_TUTORIAL_MODE=execute` with an explicit `FI_RUN_ROOT` to run the
production stages below. Execute notebooks **00 → 04** in order for the
full Experiment 0; each uses a fresh kernel and the same run directory.
Use the [notebook HAIC launchers](../../../../../slurm/future-innovation/NOTEBOOKS.md)
for scheduled execution. Inspection remains read-only. An absent local
file says nothing about the current state of a remote HAIC job.

[Study overview](../../../../../docs/studies/future-innovation/README.md) ·
[Direct gate specification](../../../../../docs/studies/future-innovation/direct-gate-protocol.md)

In [1]:
from pathlib import Path
import os
import sys
from time import perf_counter

started = perf_counter()
override = os.environ.get("GAVD6_ROOT")
if override:
    candidates = [Path(override).expanduser().resolve()]
else:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base, base / "gavd6", base / "experiments/sjepa/gavd6"))
PROJECT_ROOT = next((p for p in candidates if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the checkout containing src/gavd6_sjepa.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib_inline.backend_inline import set_matplotlib_formats
get_ipython().run_line_magic("matplotlib", "inline")
set_matplotlib_formats("svg", "png")
plt.rcParams.update({"figure.figsize": (8, 3), "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 11})

from gavd6_sjepa.research_directions.future_innovation.fi_tutorial_inspection import (
    artifact_inventory, inspect_report, read_optional_table, inspection_audit_path,
)

# Use "execute" for real stages or "inspect" for saved artifacts.
# Relative paths resolve from GAVD6_ROOT. Execution requires an explicit run root.
MODE = os.environ.get("FI_TUTORIAL_MODE", "teach")
if MODE not in {"teach", "inspect", "execute"}:
    raise ValueError("FI_TUTORIAL_MODE must be teach, inspect, or execute.")
if MODE == "execute" and not os.environ.get("FI_RUN_ROOT"):
    raise ValueError("Set FI_RUN_ROOT explicitly before executing real experiment stages.")
RUN_ROOT = Path(os.environ.get("FI_RUN_ROOT", "outputs/future-innovation/direct-v2")).expanduser()
if not RUN_ROOT.is_absolute():
    RUN_ROOT = PROJECT_ROOT / RUN_ROOT
RUN_ROOT = RUN_ROOT.resolve()
print("Teaching examples only; no empirical gait findings." if MODE == "teach"
      else f"{MODE.upper()} mode: {RUN_ROOT}")
if MODE == "execute":
    from gavd6_sjepa.research_directions.future_innovation.fi_notebook_workflow import (
        initialize_from_environment, run_stage, build_notebook_report, finish_notebook_report,
        attempt_stage, require_stage_success,
    )
    if (RUN_ROOT / "config/run-contract.json").is_file():
        import json
        saved_run = json.loads((RUN_ROOT / "config/run-contract.json").read_text())
        print("Frozen protocol:", saved_run.get("protocol", "legacy-v1"),
              "— gate clips:", saved_run.get("cohort_size"))
        if saved_run.get("protocol", "legacy-v1") == "legacy-v1":
            print("This run retains legacy selectivity gates. Use a new run root for direct-v2.")

EXECUTE mode: /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/outputs/future-innovation/gate-v2


## Execute this stage

Initialize the immutable protocol and input provenance, or validate the existing run. New runs use the same environment variables and five official annotation partitions as Slurm job 01. Complete the HAIC environment and input setup first.

This cell runs only in `execute` mode. Each command uses this kernel's Python and the existing production CLI; stage logs are retained alongside the executed notebook.

In [2]:
if MODE == "execute":
    initialize_from_environment(RUN_ROOT)

$ /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/.venv/bin/python3 -u -m gavd6_sjepa.command_line_interface future-innovation init-run --protocol direct-v2 --sequence-manifest /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/data/gavd_full/manifests/gavd_full_sequences.csv --video-manifest /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/data/gavd_full/manifests/gavd_full_videos.csv --annotations /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/data/gavd_full/annotations/GAVD/data/GAVD_Clinical_Annotations_1.csv /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/data/gavd_full/annotations/GAVD/data/GAVD_Clinical_Annotations_2.csv /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/data/gavd_full/annotations/GAVD/data/GAVD_Clinical_Annotations_3.csv /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/data/gavd_full/annotations/GAVD/data/GAVD_Clinical_Annotations_4.csv /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/data/gavd_full/annotations/GAVD/data/GAVD_Cli

Preregistered /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/outputs/future-innovation/gate-v2


## 1. Predict first, then learn what was missed

Let `x` contain past video features and recording details, `s` contain
past skeleton coordinates, confidence and validity, and `y` be the future
target. Fit a baseline using `x`. The residual `y - baseline(x)` is what
remains unexplained. A small head uses `s` and `x` to predict a correction.
It never receives `y` when making a prediction.

For one imaginary feature, an answer of 0.8 and baseline of 0.5 leave
a residual of 0.3. A predicted correction of 0.2 moves the prediction to
0.7. The example below makes that arithmetic visible.

In [3]:
if MODE == "teach":
    answer, baseline_prediction, correction = 0.8, 0.5, 0.2
    full_prediction = baseline_prediction + correction
    display(pd.DataFrame({"value": [answer, baseline_prediction, full_prediction]},
                         index=["Answer", "Video baseline", "With correction"]))
    fig, ax = plt.subplots()
    ax.barh(["Video baseline", "With correction"],
            [baseline_prediction, full_prediction], color=["#2f6f99", "#5f9e7e"])
    ax.axvline(answer, color="#e07a4b", label="Illustrative answer")
    ax.set(xlim=(0, 1), xlabel="One feature value", title="A correction can reduce the remaining error")
    ax.legend(); plt.show()

## 2. Measure the gain against a common reference

Predictive R² compares squared error with predicting the training mean.
It can be negative. `delta_r2` is full R² minus baseline R²; 0.05 means
an absolute increase of 0.05. `f8` is the fraction of the baseline's
remaining error recovered at the eight-frame horizon.

Use the production scoring function on already training-centered toy
targets. Zero therefore represents the training-mean reference. Real
fits learn this centering and scaling separately inside training sources.

In [4]:
if MODE == "teach":
    from gavd6_sjepa.research_directions.future_innovation.fi_metrics import score_arrays
    y = np.array([[-1.0], [1.0], [-0.5], [0.5]])
    scores, *_ = score_arrays(y, 0.4 * y, 0.7 * y, np.ones(4), np.array([True]))
    display(pd.DataFrame([scores])[["r2_baseline", "r2_full", "delta_r2", "f8"]])

## 3. Ask which alternative explanation each control tests

| Arm | Question |
| --- | --- |
| Real skeleton | Does the correctly paired history help? |
| Time shuffle | Does the original temporal order matter? |
| Clip mismatch | Does this skeleton need to belong to this video? |
| No skeleton | Could capacity, baseline features, or validity explain the gain? |

The last control retains validity while zeroing coordinates and
confidence. Background features still attend to the person in the
teacher. Each control narrows the interpretation; none proves a clinical
or causal mechanism on its own.

In [5]:
if MODE != "teach":
    display(artifact_inventory(RUN_ROOT))
    evidence = inspect_report(RUN_ROOT)
    print(evidence["state"], "—", evidence["explanation"])

,stage,record,present_locally
0,Setup,config/run-contract.json,True
1,Candidates,config/candidates-contract.json,False
2,Aligned cohort,config/cohort-contract.json,False
3,Teacher cache,config/cache-contract.json,False
4,Validity audits,qc/readiness-summary.json,False
5,Outer fold 0,models/fold-0/fold-complete.json,False
6,Outer fold 1,models/fold-1/fold-complete.json,False
7,Outer fold 2,models/fold-2/fold-complete.json,False
8,Outer fold 3,models/fold-3/fold-complete.json,False
9,Outer fold 4,models/fold-4/fold-complete.json,False


UNAVAILABLE — No gate decision is available in this local copy. Inspect the inventory or copy the run reports from HAIC.


## What this step establishes

Execution initializes or verifies the run's frozen protocol; teaching mode constructs an illustrative gain. Neither establishes that skeletons help on GAVD. That question requires aligned examples, source-held-out predictions and the timing, pairing and matched-capacity controls. Next we establish the cohort.

Continue with [01_cohort_and_alignment.ipynb](01_cohort_and_alignment.ipynb).

In [6]:
print(f"Notebook elapsed time: {perf_counter() - started:.2f} seconds ({MODE} mode).")

Notebook elapsed time: 60.66 seconds (execute mode).
